<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/triton_Mamba_topo_cbp_gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



| Missing Architecture Family | Representative Open Models | Core Mechanism | GGUF / `llama.cpp` Readiness | Why It Matters for Your Registry |
| --- | --- | --- | --- | --- |
| **1. Pure State Space Models (SSMs)** | **Mamba-1 / Mamba-2** (`mamba-2.8b`, `Codestral-Mamba-7B`), **Falcon-Mamba-7B** | Selective structured state space recurrence ($O(N)$ compute, constant $O(1)$ state memory; zero KV cache) | **Full native support** | Tests topological drift governance on recurrent hidden-state matrices ($h_t = \bar{A}h_{t-1} + \bar{B}x_t$) rather than attention KV caches. |


## setup

In [ ]:
!pip install -U bitsandbytes>=0.46.1 -q

In [1]:
!nvidia-smi

Mon Nov 23 12:42:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/

total 420G
drwx------ 6 root root 4.0K Nov 23 12:42 ..
-rw------- 1 root root  32G Nov 23 11:42 granite-4.0-h-small-topo-triton-Q8_0.gguf
-rw------- 1 root root  16G Nov 23 05:21 kimi-vl-ft-Q8_0.gguf
-rw------- 1 root root 3.7K Nov 22 15:34 GLM-4-32B-0414-topo-triton-Q8_0_audit.json
-rw------- 1 root root  33G Nov 22 14:19 GLM-4-32B-0414-topo-triton-Q8_0.gguf
-rw------- 1 root root 8.9K Nov 21 14:30 somala_drive_gguf_audit.json
-rw------- 1 root root  26G Nov 21 05:08 Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf
-rw------- 1 root root 7.2G Nov 19 03:18 mistral-7b-text-to-sql-topo-Q8_0.gguf
-rw------- 1 root root  27G Nov 14 20:14 qwen3.8-27b-topocbp-Q8_0.gguf
-rw------- 1 root root  11G Nov 12 06:55 deepseek_moe_16b_topocbp_q4_k_m.gguf
-rw------- 1 root root  70G Nov  9 06:01 deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf
-rw------- 1 root root  32G Nov  9 05:57 sarvam-30b-topo-governed-q8_0.gguf
-rw------- 1 root root  16G Nov  9 04:40 muse-glimmer-30b-card-faithful-Q4_K_M.ggu

In [6]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 MB 135.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.38-py3-none-linux_x86_64.whl size=30408185 sha256=0d222340f31cc3fa41c34e1e45d527c7c0ec7a41e6a6906442525883bd9e2143
  Stored in directory: /tmp/pip-ephem-wheel-cache-q91vh6ax/wheels/58/5d/66/054fd29778e664a3d60ef238401ba1dbe164d6bdfb2f406477
Successfully built llama-cpp-python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [ ]:
!pip show transformers llama-cpp-python bitsandbytes

Name: transformers
Version: 5.22.1
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.13/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: peft, sentence-transformers
---
Name: llama_cpp_python
Version: 0.3.37
Summary: Python bindings for the llama.cpp library
Home-page: https://github.com/abetlen/llama-cpp-python
Author: 
Author-email: Andrei Betlen <abetlen@gmail.com>
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: diskcache, jinja2, numpy, typing-extension

## train

In [ ]:
import os
import warnings

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
warnings.filterwarnings("ignore", category=FutureWarning, message=r".*rope_scaling.*")

import gc
import math
import hashlib
import random
import torch
from torch.optim import Optimizer
from torch.utils.data import DataLoader
import transformers
from packaging.version import Version
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    get_cosine_schedule_with_warmup,
)
from transformers.utils import logging as hf_logging
from datasets import load_dataset

hf_logging.set_verbosity_error()

try:
    import triton
    import triton.language as tl

    HAS_TRITON = True
except ImportError:
    triton = None
    tl = None
    HAS_TRITON = False

try:
    import mamba_ssm  # noqa: F401

    HAS_MAMBA_SSM = True
except ImportError:
    HAS_MAMBA_SSM = False

# =====================================================================
# 0. SEED & DEVICE
# =====================================================================
SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
use_cuda = num_gpus > 0
if not use_cuda:
    raise RuntimeError("4-bit bitsandbytes loading requires a CUDA GPU.")
primary_device = torch.device("cuda:0")

# =====================================================================
# 1. OPTIMIZER: TopoAdamW with bf16 states, stochastic rounding and
#    error-compensated bf16 parameter updates (fused Triton kernel +
#    PyTorch fallback). Optional prime anchor rows for trained embeddings.
# =====================================================================
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]


def _stochastic_round_(src_fp32: torch.Tensor, dst_bf16: torch.Tensor) -> None:
    """Round fp32 -> bf16 stochastically. NOTE: src_fp32 is modified in place."""
    bits = src_fp32.view(torch.int32)
    bits.add_(torch.randint_like(bits, 0, 1 << 16))
    bits.bitwise_and_(-65536)
    dst_bf16.copy_(src_fp32)


if HAS_TRITON:

    @triton.jit
    def _topo_adamw_fused_step_kernel(
        p_ptr,
        grad_ptr,
        m_buf_ptr,
        v_buf_ptr,
        comp_ptr,
        n_elements,
        lr,
        beta1,
        beta2,
        eps,
        wd,
        bias_corr1,
        bias_corr2,
        seed_m,
        seed_v,
        P_IS_BF16: tl.constexpr,
        BLOCK_SIZE: tl.constexpr,
    ):
        pid = tl.program_id(axis=0)
        offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n_elements

        p_raw = tl.load(p_ptr + offsets, mask=mask, other=0.0)
        g = tl.load(grad_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        m_prev = tl.load(m_buf_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        v_prev = tl.load(v_buf_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
        comp_prev = tl.load(comp_ptr + offsets, mask=mask, other=0.0).to(tl.float32)

        w = p_raw.to(tl.float32)

        m = beta1 * m_prev + (1.0 - beta1) * g
        v = beta2 * v_prev + (1.0 - beta2) * g * g

        m_hat = m / bias_corr1
        v_hat = v / bias_corr2
        denom = tl.sqrt(v_hat) + eps
        step_update = -lr * (m_hat / denom + wd * w) + comp_prev

        total = w + step_update
        if P_IS_BF16:
            new_p = total.to(tl.bfloat16)
            new_comp = total - new_p.to(tl.float32)
            tl.store(p_ptr + offsets, new_p, mask=mask)
            tl.store(comp_ptr + offsets, new_comp.to(tl.bfloat16), mask=mask)
        else:
            tl.store(p_ptr + offsets, total, mask=mask)
            tl.store(comp_ptr + offsets, tl.zeros_like(total).to(tl.bfloat16), mask=mask)

        # Stochastic rounding of the moments to bf16. Jitter is cast to int32
        # explicitly because tl.randint returns uint32.
        m_bits = m.to(tl.int32, bitcast=True)
        m_jitter = (tl.randint(seed_m, offsets) & 0xFFFF).to(tl.int32)
        m_rounded_bits = (m_bits + m_jitter) & -65536
        m_rounded = m_rounded_bits.to(tl.float32, bitcast=True).to(tl.bfloat16)

        v_bits = v.to(tl.int32, bitcast=True)
        v_jitter = (tl.randint(seed_v, offsets) & 0xFFFF).to(tl.int32)
        v_rounded_bits = (v_bits + v_jitter) & -65536
        v_rounded = v_rounded_bits.to(tl.float32, bitcast=True).to(tl.bfloat16)

        tl.store(m_buf_ptr + offsets, m_rounded, mask=mask)
        tl.store(v_buf_ptr + offsets, v_rounded, mask=mask)


def _triton_param_step(
    p,
    grad,
    m_buf,
    v_buf,
    comp,
    lr,
    beta1,
    beta2,
    eps,
    wd,
    step_t,
    base_seed=SEED,
    param_id=0,
    block_size=1024,
) -> None:
    n_elements = p.numel()
    if n_elements == 0:
        return

    p_c = p.contiguous().view(-1)
    g_c = grad.contiguous().view(-1)
    m_c = m_buf.contiguous().view(-1)
    v_c = v_buf.contiguous().view(-1)
    c_c = comp.contiguous().view(-1)

    bias_corr1 = 1.0 - (beta1**step_t)
    bias_corr2 = 1.0 - (beta2**step_t)
    seed_m = int((base_seed * 1_000_003 + step_t * 97 + param_id * 7919) & 0x7FFFFFFF)
    seed_v = int((base_seed * 1_000_033 + step_t * 193 + param_id * 104729) & 0x7FFFFFFF)

    grid = (triton.cdiv(n_elements, block_size),)
    _topo_adamw_fused_step_kernel[grid](
        p_c,
        g_c,
        m_c,
        v_c,
        c_c,
        n_elements,
        float(lr),
        float(beta1),
        float(beta2),
        float(eps),
        float(wd),
        float(bias_corr1),
        float(bias_corr2),
        seed_m,
        seed_v,
        P_IS_BF16=(p_c.dtype == torch.bfloat16),
        BLOCK_SIZE=block_size,
    )

    if not p.is_contiguous():
        p.copy_(p_c.view_as(p))
    if not m_buf.is_contiguous():
        m_buf.copy_(m_c.view_as(m_buf))
    if not v_buf.is_contiguous():
        v_buf.copy_(v_c.view_as(v_buf))
    if not comp.is_contiguous():
        comp.copy_(c_c.view_as(comp))


def _pytorch_param_step(p, m_buf, v_buf, comp, lr, beta1, beta2, eps, wd, step_t) -> None:
    g = p.grad.float()
    m = m_buf.float().mul_(beta1).add_(g, alpha=1.0 - beta1)
    v = v_buf.float().mul_(beta2).addcmul_(g, g, value=1.0 - beta2)
    del g

    w = p.detach().float().clone()
    denom = (v.sqrt() / math.sqrt(1.0 - beta2**step_t)).add_(eps)
    y = m.div(denom).mul_(-lr / (1.0 - beta1**step_t))
    del denom

    y.add_(w, alpha=-lr * wd)
    if p.dtype == torch.bfloat16:
        y.add_(comp.float())
    _stochastic_round_(m, m_buf)
    _stochastic_round_(v, v_buf)
    del m, v

    total = w.add_(y)
    if p.dtype == torch.bfloat16:
        new = total.to(torch.bfloat16)
        comp.copy_(total.sub_(new.float()))
        p.copy_(new)
    else:
        p.copy_(total.to(p.dtype))
        comp.zero_()
    del y, w, total


class TopoAdamW(Optimizer):
    """AdamW with bf16 moments (stochastic rounding) and compensated bf16 params."""

    def __init__(
        self,
        params,
        lr=1e-4,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-2,
        embedding_layer=None,
        prime_anchors=None,
        base_seed=SEED,
        use_triton=True,
    ):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))
        self.base_seed = base_seed
        self.use_triton = bool(use_triton and HAS_TRITON)
        self.embedding_layer = embedding_layer

        self._param_ids = {}
        for g in self.param_groups:
            for p in g["params"]:
                self._param_ids[id(p)] = len(self._param_ids)

        self.anchors_active = False
        self.valid_anchors = []
        self.baseline = {}
        self.certified_sha256_digest = None

        if embedding_layer is not None:
            emb_w = embedding_layer.weight
            optimized = any(p is emb_w for g in self.param_groups for p in g["params"])
            if optimized:
                anchors = list(prime_anchors) if prime_anchors is not None else list(PRIME_ANCHORS)
                self.valid_anchors = [i for i in anchors if i < emb_w.shape[0]]
                self.baseline = {i: emb_w.data[i].detach().clone() for i in self.valid_anchors}
                self.certified_sha256_digest = self._digest()
                self.anchors_active = bool(self.valid_anchors)

        print(
            f"[TopoAdamW] Triton={self.use_triton} | anchors_active={self.anchors_active}"
            + (f" | digest {self.certified_sha256_digest[:16]}" if self.anchors_active else "")
        )

    def _digest(self) -> str:
        w = self.embedding_layer.weight.data
        buf = torch.cat([w[i].flatten().float() for i in self.valid_anchors]).cpu().numpy().tobytes()
        return hashlib.sha256(buf).hexdigest()

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        emb_w = self.embedding_layer.weight if self.anchors_active else None
        if emb_w is not None and emb_w.grad is not None:
            for i in self.valid_anchors:
                emb_w.grad[i].zero_()

        for group in self.param_groups:
            lr, wd, eps = group["lr"], group["weight_decay"], group["eps"]
            beta1, beta2 = group["betas"]
            for p in group["params"]:
                if not p.requires_grad or p.grad is None:
                    continue
                state = self.state[p]
                if not state:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p, dtype=torch.bfloat16)
                    state["exp_avg_sq"] = torch.zeros_like(p, dtype=torch.bfloat16)
                    state["comp"] = torch.zeros_like(p, dtype=torch.bfloat16)

                state["step"] += 1
                t = state["step"]
                m_buf, v_buf, comp = state["exp_avg"], state["exp_avg_sq"], state["comp"]
                grad_on_dev = p.grad.to(p.device) if p.grad.device != p.device else p.grad

                if self.use_triton and p.is_cuda and grad_on_dev.is_cuda:
                    _triton_param_step(
                        p,
                        grad_on_dev,
                        m_buf,
                        v_buf,
                        comp,
                        lr,
                        beta1,
                        beta2,
                        eps,
                        wd,
                        t,
                        base_seed=self.base_seed,
                        param_id=self._param_ids[id(p)],
                    )
                else:
                    orig_grad = p.grad
                    p.grad = grad_on_dev
                    try:
                        _pytorch_param_step(p, m_buf, v_buf, comp, lr, beta1, beta2, eps, wd, t)
                    finally:
                        p.grad = orig_grad

                if emb_w is not None and p is emb_w:
                    for i in self.valid_anchors:
                        p[i].copy_(self.baseline[i].to(device=p.device, dtype=p.dtype))
                        comp[i].zero_()

        return loss

    def audit_telemetry(self):
        """Returns (max_drift, digest, matched), or (None, None, None) if anchoring is inactive."""
        if not self.anchors_active:
            return None, None, None
        w = self.embedding_layer.weight.data
        drift = max(
            torch.max(torch.abs(w[i].float() - self.baseline[i].to(w.device).float())).item()
            for i in self.valid_anchors
        )
        h = self._digest()
        return drift, h, h == self.certified_sha256_digest


def selftest_optimizer():
    if not (HAS_TRITON and torch.cuda.is_available()):
        print("[selftest] skipped (needs Triton + CUDA)")
        return
    for dtype in (torch.bfloat16, torch.float32):
        torch.manual_seed(0)
        shape = (257, 129)
        p0 = torch.randn(shape, device="cuda").to(dtype)
        g = torch.randn(shape, device="cuda").to(torch.bfloat16)
        outs = []
        for use_t in (True, False):
            p = p0.clone()
            p.grad = g.clone()
            m = torch.zeros(shape, device="cuda", dtype=torch.bfloat16)
            v = torch.zeros_like(m)
            c = torch.zeros_like(m)
            if use_t:
                _triton_param_step(p, g, m, v, c, 1e-3, 0.9, 0.999, 1e-8, 1e-2, 1)
            else:
                _pytorch_param_step(p, m, v, c, 1e-3, 0.9, 0.999, 1e-8, 1e-2, 1)
            outs.append(p.float())
        diff = (outs[0] - outs[1]).abs().max().item()
        print(f"[selftest] {dtype}: max |triton - pytorch| = {diff:.3e}")


if os.environ.get("RUN_OPT_SELFTEST") == "1":
    selftest_optimizer()

# =====================================================================
# 2. MODEL / TOKENIZER (ibm-granite/granite-4.0-h-small: Mamba-2 + MoE Hybrid)
# =====================================================================
model_id = "ibm-granite/granite-4.0-h-small"
max_seq_len = 512

config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

has_prequant = getattr(config, "quantization_config", None) is not None
model_kwargs = dict(
    config=config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
if not has_prequant:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_skip_modules=["lm_head"],
    )

print(
    f"[*] transformers {transformers.__version__} | mamba_ssm={HAS_MAMBA_SSM} | "
    f"model_type={getattr(config, 'model_type', 'unknown')}"
)
_dtype_kw = "dtype" if Version(transformers.__version__) >= Version("4.56.0") else "torch_dtype"
try:
    model = AutoModelForCausalLM.from_pretrained(
        model_id, **{_dtype_kw: torch.bfloat16}, **model_kwargs
    )
except TypeError as e:
    if "unexpected keyword argument" not in str(e) or _dtype_kw not in str(e):
        raise
    _dtype_kw = "torch_dtype" if _dtype_kw == "dtype" else "dtype"
    print(f"[warn] retrying with {_dtype_kw}=")
    model = AutoModelForCausalLM.from_pretrained(
        model_id, **{_dtype_kw: torch.bfloat16}, **model_kwargs
    )

model.config.use_cache = False
gc.collect()
torch.cuda.empty_cache()

embedding_layer = model.get_input_embeddings()

# ---- Choose trainable params for Hybrid Mamba-2 + MoE ----
# Set TRAIN_MAMBA_AND_ROUTER=1 to also train Mamba-2 SSM/conv/dt/in_proj/out_proj
# and MoE router gates alongside lm_head and normalization layers.
train_mamba_and_router = os.environ.get("TRAIN_MAMBA_AND_ROUTER", "0") == "1"

for p in model.parameters():
    p.requires_grad = False

decay_params, no_decay_params, seen = [], [], set()
for name, mod in model.named_modules():
    is_head = name.endswith("lm_head")
    is_norm = "norm" in name.lower()
    is_mamba_or_gate = train_mamba_and_router and any(
        tok in name.lower() for tok in ("mamba", "mixer", "conv1d", "dt_proj", "router", "gate")
    )
    if not (is_head or is_norm or is_mamba_or_gate):
        continue
    for p in mod.parameters(recurse=False):
        if id(p) in seen or not p.is_floating_point():
            continue
        seen.add(id(p))
        p.requires_grad = True
        use_decay = is_head and p.ndim >= 2
        (decay_params if use_decay else no_decay_params).append(p)

trainable_params = decay_params + no_decay_params
if not trainable_params:
    raise RuntimeError("No trainable parameters found; check module names.")

if hasattr(model, "gradient_checkpointing_enable"):
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except Exception:
        model.gradient_checkpointing_enable()

try:
    model.enable_input_require_grads()
except Exception as e:
    print(f"[warn] enable_input_require_grads failed: {e!r}")

print(
    f"[*] Trainable parameters: {sum(p.numel() for p in trainable_params):,} "
    f"(train_mamba_and_router={train_mamba_and_router})"
)

WEIGHT_DECAY = 1e-2
topo_optimizer = TopoAdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=1e-4,
    embedding_layer=embedding_layer,
    prime_anchors=PRIME_ANCHORS,
    base_seed=SEED,
    use_triton=True,
)


def _get_input_device() -> torch.device:
    emb = embedding_layer
    if emb is not None and hasattr(emb, "weight") and emb.weight.device.type != "meta":
        return emb.weight.device
    return primary_device


input_device = _get_input_device()
print(f"[*] Input target device: {input_device}")

# =====================================================================
# 3. DATA
# =====================================================================
_im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
if _im_end_id is not None and _im_end_id != tokenizer.unk_token_id:
    END_TOKEN = "<|im_end|>"
else:
    END_TOKEN = tokenizer.eos_token or ""
print(f"[*] End-of-turn token used for labels: {END_TOKEN!r}")


def build_user_prompt(sample):
    instruction = (sample.get("instruction") or "").strip()
    extra_input = (sample.get("input") or "").strip()
    return f"{instruction}\n\nInput:\n{extra_input}" if extra_input else instruction


def preprocess_function(sample):
    user = build_user_prompt(sample)
    if getattr(tokenizer, "chat_template", None):
        try:
            prompt_text = tokenizer.apply_chat_template(
                [{"role": "user", "content": user}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            prompt_text = tokenizer.apply_chat_template(
                [{"role": "user", "content": user}],
                tokenize=False,
                add_generation_prompt=True,
            )
    else:
        bos = tokenizer.bos_token or ""
        prompt_text = f"{bos}{user}\n"

    answer_text = (sample.get("output") or "") + END_TOKEN
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer_text, add_special_tokens=False)["input_ids"]
    input_ids = (list(prompt_ids) + answer_ids)[:max_seq_len]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:max_seq_len]
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}


def has_supervised_tokens(ex):
    return any(l != -100 for l in ex["labels"])


raw = load_dataset("yahma/alpaca-cleaned", split="train").shuffle(seed=SEED)
train_ds = (
    raw.select(range(12500))
    .map(preprocess_function, remove_columns=raw.column_names)
    .filter(has_supervised_tokens)
)
eval_ds = (
    raw.select(range(12500, 13000))
    .map(preprocess_function, remove_columns=raw.column_names)
    .filter(has_supervised_tokens)
)
print(f"[*] train samples: {len(train_ds)} | eval samples: {len(eval_ds)}")

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
)
pin = use_cuda and input_device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collator, pin_memory=pin)
eval_loader = DataLoader(eval_ds, batch_size=2, shuffle=False, collate_fn=collator, pin_memory=pin)

# =====================================================================
# 4. TRAINING & AUDIT LOOP
# =====================================================================
epochs = 3
grad_accum = 16
max_grad_norm = 1.0
total_steps = (len(train_loader) * epochs) // grad_accum
scheduler = get_cosine_schedule_with_warmup(
    topo_optimizer, num_warmup_steps=20, num_training_steps=total_steps
)


@torch.no_grad()
def evaluate_loss():
    model.eval()
    losses = []
    for batch in eval_loader:
        batch = {k: v.to(input_device, non_blocking=pin) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            out = model(**batch)
        losses.append(out.loss.item())
    model.train()
    return sum(losses) / max(len(losses), 1)


def save_trainable(path):
    state = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
    torch.save(state, os.path.join(path, "trainable_weights.pt"))


print(f"[baseline] validation loss before training: {evaluate_loss():.4f}", flush=True)

# Mount Drive first in Colab: from google.colab import drive; drive.mount("/content/drive")
output_dir = "/content/drive/MyDrive/topogpt/granite-4.0-h-small-alpaca-topo-4bit-triton-train"
os.makedirs(output_dir, exist_ok=True)

model.train()
topo_optimizer.zero_grad()
global_step = 0
micro_step = 0
running_step_loss = 0.0
accum_loss = 0.0

for epoch in range(1, epochs + 1):
    for batch in train_loader:
        batch = {k: v.to(input_device, non_blocking=pin) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / grad_accum

        loss.backward()
        accum_loss += outputs.loss.detach().item() / grad_accum
        micro_step += 1

        if micro_step % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
            topo_optimizer.step()
            scheduler.step()
            topo_optimizer.zero_grad()

            global_step += 1
            running_step_loss += accum_loss
            accum_loss = 0.0

            if global_step % 20 == 0:
                avg_loss = running_step_loss / 20.0
                lr_now = scheduler.get_last_lr()[0]
                print(f"step {global_step:>5} | train loss {avg_loss:.4f} | lr {lr_now:.2e}", flush=True)
                running_step_loss = 0.0

            if global_step % 100 == 0:
                print(f"[eval] step {global_step:>5} | val loss {evaluate_loss():.4f}", flush=True)

    epoch_dir = os.path.join(output_dir, f"epoch-{epoch}")
    os.makedirs(epoch_dir, exist_ok=True)
    save_trainable(epoch_dir)
    print(f"[*] Saved epoch {epoch} trainable weights -> {epoch_dir}", flush=True)

max_drift, current_hash, hash_matched = topo_optimizer.audit_telemetry()
if max_drift is None:
    print("[TopoAudit] anchors inactive (embedding is frozen), nothing to audit.")
else:
    print(f"[TopoAudit] Max Anchor Drift: {max_drift:.6e} | Hash Matched: {hash_matched} | Digest: {current_hash[:16]}")

save_trainable(output_dir)
tokenizer.save_pretrained(output_dir)
print("[*] Done.")

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.15M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/6.42k [00:00<?, ?B/s]

[*] transformers 5.22.1 | mamba_ssm=False | model_type=granitemoehybrid


... (12 outputs left out)

Filter:   0%|          | 0/12500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

[*] train samples: 12500 | eval samples: 500
[baseline] validation loss before training: 0.9974
step    20 | train loss 0.9504 | lr 1.00e-04
step    40 | train loss 0.8992 | lr 9.99e-05
step    60 | train loss 0.8839 | lr 9.97e-05
step    80 | train loss 0.8439 | lr 9.93e-05
step   100 | train loss 0.8395 | lr 9.88e-05
[eval] step   100 | val loss 0.8480
step   120 | train loss 0.8242 | lr 9.81e-05
step   140 | train loss 0.8278 | lr 9.73e-05
step   160 | train loss 0.8653 | lr 9.64e-05
step   180 | train loss 0.8715 | lr 9.53e-05
step   200 | train loss 0.8202 | lr 9.41e-05
[eval] step   200 | val loss 0.8423
step   220 | train loss 0.8372 | lr 9.27e-05
step   240 | train loss 0.8506 | lr 9.13e-05
step   260 | train loss 0.8426 | lr 8.97e-05
step   280 | train loss 0.8392 | lr 8.79e-05
step   300 | train loss 0.8307 | lr 8.61e-05
[eval] step   300 | val loss 0.8389
step   320 | train loss 0.8132 | lr 8.42e-05
step   340 | train loss 0.8312 | lr 8.21e-05
step   360 | train loss 0.8236 

In [ ]:
%cd /content/drive/MyDrive/topogpt/granite-4.0-h-small-alpaca-topo-4bit-triton-train
!du -sh *

/content/drive/MyDrive/topogpt/granite-4.0-h-small-alpaca-topo-4bit-triton-train
6.5K	chat_template.jinja
786M	epoch-1
786M	epoch-2
786M	epoch-3
512	tokenizer_config.json
6.9M	tokenizer.json
786M	trainable_weights.pt


## merger

In [ ]:
import os
import hashlib
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_ID = "ibm-granite/granite-4.0-h-small"
TRAIN_DIR = "/content/drive/MyDrive/topogpt/granite-4.0-h-small-alpaca-topo-4bit-triton-train"
MERGED_DIR = "/content/granite-4.0-h-small-topo-merged-bf16"
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]

# 1. Load base model in full bfloat16 on CPU (avoid 4-bit quantization for GGUF export)
print("[1/4] Loading base model in bfloat16 on CPU...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(TRAIN_DIR, trust_remote_code=True)

# 2. Load and apply your trained weights (lm_head + norms)
print("[2/4] Applying trainable_weights.pt...")
trainable_state = torch.load(
    os.path.join(TRAIN_DIR, "trainable_weights.pt"),
    map_location="cpu",
    weights_only=True,
)
missing, unexpected = model.load_state_dict(trainable_state, strict=False)
print(f"Applied {len(trainable_state)} tensors | Unexpected keys: {len(unexpected)}")

# 3. Verify prime anchor integrity before export
emb_w = model.get_input_embeddings().weight.data
buf = torch.cat([emb_w[i].flatten().float() for i in PRIME_ANCHORS]).numpy().tobytes()
digest = hashlib.sha256(buf).hexdigest()
print(f"[3/4] Verified Anchor SHA-256 prefix: {digest[:16]}")

# 4. Save merged Hugging Face directory (safetensors + config + tokenizer)
print(f"[4/4] Saving merged BF16 model to {MERGED_DIR}...")
os.makedirs(MERGED_DIR, exist_ok=True)
model.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="5GB")
tokenizer.save_pretrained(MERGED_DIR)
print("[*] Merged directory ready for llama.cpp conversion.")

[1/4] Loading base model in bfloat16 on CPU...


Loading weights:   0%|          | 0/586 [00:00<?, ?it/s]

[2/4] Applying trainable_weights.pt...
Applied 118 tensors | Unexpected keys: 0
[3/4] Verified Anchor SHA-256 prefix: 3a5248c8273eba5e
[4/4] Saving merged BF16 model to /content/granite-4.0-h-small-topo-merged-bf16...


Writing model shards:   0%|          | 0/14 [00:00<?, ?it/s]

[*] Merged directory ready for llama.cpp conversion.


## gguf-convertor

In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/gguf/

total 12K
drwx------ 29 root root 4.0K Nov 23 07:26 ..
drwx------  2 root root 4.0K Nov 21 04:59 nemotron
drwx------  3 root root 4.0K Oct 26 15:10 .


In [ ]:
import os
import subprocess

LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
MERGED = "/content/granite-4.0-h-small-topo-merged-bf16"
GGUF_BF16 = "/content/drive/MyDrive/topogpt/gguf/granite-4.0-h-small-topo-triton-bf16.gguf"

subprocess.run(
    [
        "python",
        f"{LLAMA_CPP_DIR}/convert_hf_to_gguf.py",
        MERGED,
        "--outfile",
        GGUF_BF16,
        "--outtype",
        "bf16",
    ],
    check=True,
)

print(GGUF_BF16, f"{os.path.getsize(GGUF_BF16) / 1e9:.1f} GB")

/content/drive/MyDrive/topogpt/gguf/granite-4.0-h-small-topo-triton-bf16.gguf 64.4 GB


In [ ]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize

-rwx------ 1 root root 18K Nov  2 10:56 /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize


## gguf-quantize

In [ ]:
!pip install --upgrade gguf

In [ ]:
import os
import subprocess

LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
GGUF_BF16 = "/content/drive/MyDrive/topogpt/gguf/granite-4.0-h-small-topo-triton-bf16.gguf"
GGUF_Q8   = "/content/drive/MyDrive/GGUF_Models/granite-4.0-h-small-topo-triton-Q8_0.gguf"

os.makedirs(os.path.dirname(GGUF_Q8), exist_ok=True)

BIN_DIR = f"{LLAMA_CPP_DIR}/build/bin"
BIN = f"{BIN_DIR}/llama-quantize"
env = {**os.environ, "LD_LIBRARY_PATH": f"{BIN_DIR}:{os.environ.get('LD_LIBRARY_PATH', '')}"}

if not os.path.exists(BIN):
    subprocess.run(
        f"cd {LLAMA_CPP_DIR} && cmake -B build -DLLAMA_CURL=OFF && cmake --build build -j --target llama-quantize",
        shell=True,
        check=True,
    )

subprocess.run([BIN, GGUF_BF16, GGUF_Q8, "Q8_0"], check=True, env=env)
print(GGUF_Q8, f"{os.path.getsize(GGUF_Q8) / 1e9:.1f} GB")

/content/drive/MyDrive/GGUF_Models/granite-4.0-h-small-topo-triton-Q8_0.gguf 34.3 GB


In [3]:
from gguf import GGUFReader

GGUF = "/content/drive/MyDrive/GGUF_Models/granite-4.0-h-small-topo-triton-Q8_0.gguf"
reader = GGUFReader(GGUF)

kv = reader.fields["granitehybrid.attention.head_count_kv"]
print("head_count_kv:", [kv.parts[kv.data[i]] for i in range(len(kv.data))])
print("blk.5 tensors:", [t.name for t in reader.tensors if t.name.startswith("blk.5.")])

attn_layers = sorted({
    int(t.name.split(".")[1])
    for t in reader.tensors
    if t.name.startswith("blk.") and ".attn_q." in t.name
})
print("Attention layers found in GGUF:", attn_layers)

head_count_kv: [memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int32), memmap([0], dtype=int3

In [4]:
import os
import shutil
import numpy as np
from gguf import GGUFReader

TMP_OUT = "/content/granite-4.0-h-small-topo-triton-Q8_0-patched.gguf"
OUT = "/content/drive/MyDrive/GGUF_Models/granite-4.0-h-small-topo-triton-Q8_0-patched.gguf"

reader = GGUFReader(TMP_OUT)
kv = reader.fields["granitehybrid.attention.head_count_kv"]
patched_vals = [int(kv.parts[kv.data[i]].view(np.int32)[0]) for i in range(len(kv.data))]
print("Verified head_count_kv in patched GGUF:", patched_vals)
print("Total tensors intact:", len(reader.tensors))

shutil.move(TMP_OUT, OUT)
print("Saved clean patched file:", OUT, f"{os.path.getsize(OUT) / 1e9:.2f} GB")

Verified head_count_kv in patched GGUF: [0, 0, 0, 2048, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2048, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2048, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2048, 0, 0, 0, 0, 0, 0]
Total tensors intact: 666
Saved clean patched file: /content/drive/MyDrive/GGUF_Models/granite-4.0-h-small-topo-triton-Q8_0-patched.gguf 34.26 GB


In [5]:
from gguf import GGUFReader

SRC = "/content/drive/MyDrive/GGUF_Models/granite-4.0-h-small-topo-triton-Q8_0.gguf"
reader = GGUFReader(SRC)
kv = reader.fields["granitehybrid.attention.head_count_kv"]

print("Number of elements in head_count_kv:", len(kv.data))
for idx in [0, 5, 15, 25, 35]:
    part = kv.parts[kv.data[idx]]
    print(f"idx={idx}: dtype={part.dtype}, shape={part.shape}, data_offset={getattr(kv, 'offset', None)}")

Number of elements in head_count_kv: 40
idx=0: dtype=int32, shape=(1,), data_offset=551
idx=5: dtype=int32, shape=(1,), data_offset=551
idx=15: dtype=int32, shape=(1,), data_offset=551
idx=25: dtype=int32, shape=(1,), data_offset=551
idx=35: dtype=int32, shape=(1,), data_offset=551


In [8]:
import os
import shutil
import struct
import numpy as np
from gguf import GGUFReader

SRC = "/content/drive/MyDrive/GGUF_Models/granite-4.0-h-small-topo-triton-Q8_0.gguf"
TMP_OUT = "/content/granite-4.0-h-small-topo-triton-Q8_0-patched.gguf"
OUT = "/content/drive/MyDrive/GGUF_Models/granite-4.0-h-small-topo-triton-Q8_0-patched.gguf"

OFFSET = 612
ATTN_LAYERS = {5, 15, 25, 35}
KV_HEADS = 8

target_kv = [KV_HEADS if i in ATTN_LAYERS else 0 for i in range(40)]
patched_bytes = struct.pack("<40i", *target_kv)

shutil.copyfile(SRC, TMP_OUT)
with open(TMP_OUT, "r+b") as f:
    f.seek(OFFSET)
    f.write(patched_bytes)

reader = GGUFReader(TMP_OUT)
kv = reader.fields["granitehybrid.attention.head_count_kv"]
vals = [int(kv.parts[kv.data[i]].view(np.int32)[0]) for i in range(len(kv.data))]
print("Verified head_count_kv:", vals)
print("Total tensors intact:", len(reader.tensors))

shutil.move(TMP_OUT, OUT)
print("Saved patched GGUF:", OUT, f"{os.path.getsize(OUT) / 1e9:.2f} GB")

Verified head_count_kv: [0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0]
Total tensors intact: 666
Saved patched GGUF: /content/drive/MyDrive/GGUF_Models/granite-4.0-h-small-topo-triton-Q8_0-patched.gguf 34.26 GB


In [9]:
from llama_cpp import Llama

OUT = "/content/drive/MyDrive/GGUF_Models/granite-4.0-h-small-topo-triton-Q8_0-patched.gguf"

llm = Llama(
    model_path=OUT,
    n_ctx=1024,
    n_gpu_layers=12,
    verbose=False,
)

out = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Explain what a mixture-of-experts model is in two sentences."}],
    max_tokens=128,
    temperature=0.0,
)
print(out["choices"][0]["message"]["content"])

llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024


A mixture-of-experts model is a machine learning architecture that combines the outputs of multiple specialized models, known as "experts," to make predictions or decisions. Each expert is trained on a specific subset of the data or a particular aspect of the problem, and their outputs are combined using a gating mechanism to produce the final result.


## hf

In [10]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

HF_TOKEN = userdata.get("HF_TOKEN")
user_or_name = "frankmorales2020"

REPO_ID = f"{user_or_name}/granite-4.0-h-small-topo-triton-Q8_0-GGUF"
GGUF = "/content/drive/MyDrive/GGUF_Models/granite-4.0-h-small-topo-triton-Q8_0-patched.gguf"

login(token=HF_TOKEN)
api = HfApi()
api.create_repo(REPO_ID, repo_type="model", private=False, exist_ok=True)

README = """---
license: apache-2.0
base_model: ibm-granite/granite-4.0-h-small
tags: [gguf, llama.cpp, granite-4.0, mamba-2, moe, topo-cbp]
---
# granite-4.0-h-small Topo-CBP Fine-Tune (GGUF, Q8_0)

GGUF Q8_0 build of a fine-tuned `ibm-granite/granite-4.0-h-small` (32B hybrid Mamba-2 + Transformer + MoE)
trained on `yahma/alpaca-cleaned` with `TopoAdamW` and prime embedding anchor auditing, with
`granitehybrid.attention.head_count_kv` verified for attention layers `[5, 15, 25, 35]`.
"""

api.upload_file(
    path_or_fileobj=README.encode(),
    path_in_repo="README.md",
    repo_id=REPO_ID,
)
api.upload_file(
    path_or_fileobj=GGUF,
    path_in_repo="granite-4.0-h-small-topo-triton-Q8_0.gguf",
    repo_id=REPO_ID,
)
print(f"https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-triton-Q8_0-patched.gguf:   0%|          |  538kB / 34.3GB            

Validating                    :           |   0%            

https://huggingface.co/frankmorales2020/granite-4.0-h-small-topo-triton-Q8_0-GGUF


## inference

In [1]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

REPO_ID = "frankmorales2020/granite-4.0-h-small-topo-triton-Q8_0-GGUF"
FILENAME = "granite-4.0-h-small-topo-triton-Q8_0.gguf"

# Download / reuse cached GGUF from Hugging Face
model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    token=userdata.get("HF_TOKEN"),
)
print("Cached GGUF path:", model_path)

llm = Llama(
    model_path=model_path,
    n_ctx=1024,
    n_gpu_layers=12,  # conservative on L4 (~22.5 GB VRAM)
    verbose=False,
)

prompts = [
    "Explain what a mixture-of-experts model is in two sentences.",
    "Give a concise 3-bullet summary of why hybrid Mamba-2 and attention architectures are used.",
    "Write a short Python function that checks whether a number is prime.",
]

for i, prompt in enumerate(prompts, start=1):
    res = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=160,
        temperature=0.0,
    )
    reply = res["choices"][0]["message"]["content"].strip()
    print(f"\n--- Prompt {i} ---\n{prompt}\n--- Response {i} ---\n{reply}")

You have used a large amount of egress over the last 30 days. To protect our platform health, once you reach your account's max quota, your downloads will be throttled or blocked. If you regularly need more, you may consider upgrading your current user or organization plan to benefit from higher allowances.


granite-4.0-h-small-topo-triton-Q8_0.ggu(…): reconstructing file:   0%|          |  0.00B / 34.3GB            

granite-4.0-h-small-topo-triton-Q8_0.ggu(…): downloading bytes:           |  0.00B            

Cached GGUF path: /root/.cache/huggingface/hub/models--frankmorales2020--granite-4.0-h-small-topo-triton-Q8_0-GGUF/snapshots/af9b847e123daa4fa75acbd43a75983c5201d094/granite-4.0-h-small-topo-triton-Q8_0.gguf


llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024



--- Prompt 1 ---
Explain what a mixture-of-experts model is in two sentences.
--- Response 1 ---
A mixture-of-experts model is a machine learning architecture that combines the outputs of multiple specialized models, known as "experts," to make predictions or decisions. Each expert is trained on a specific subset of the data or a particular aspect of the problem, and their outputs are combined using a gating mechanism to produce the final result.

--- Prompt 2 ---
Give a concise 3-bullet summary of why hybrid Mamba-2 and attention architectures are used.
--- Response 2 ---
1. Enhanced Performance: Hybrid Mamba-2 and attention architectures combine the strengths of both models, resulting in improved performance in tasks such as natural language processing, image recognition, and speech recognition.

2. Efficient Resource Utilization: These architectures allow for efficient use of computational resources by leveraging the strengths of both models, reducing the need for excessive computa

In [2]:
import hashlib
import json
import os
from pathlib import Path
import numpy as np
from gguf import GGUFReader
from huggingface_hub import hf_hub_download

REPO_ID = "frankmorales2020/granite-4.0-h-small-topo-triton-Q8_0-GGUF"
FILENAME = "granite-4.0-h-small-topo-triton-Q8_0.gguf"
DRIVE_FALLBACK = "/content/drive/MyDrive/GGUF_Models/granite-4.0-h-small-topo-triton-Q8_0-patched.gguf"
REF_RECORD = Path("/content/drive/MyDrive/GGUF_Models/granite_4_0_h_small_q8_0_audit.json")

# Prefer local HF cache or Google Drive copy first to avoid extra Hub egress
local_gguf = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_files_only=True) if os.path.exists("/root/.cache/huggingface/hub") else DRIVE_FALLBACK
if not os.path.exists(local_gguf):
    local_gguf = DRIVE_FALLBACK

reader = GGUFReader(local_gguf)

# 1. Metadata verification
arch = bytes(reader.fields["general.architecture"].parts[-1]).decode("utf-8")
n_blocks = int(reader.fields["granitehybrid.block_count"].parts[-1].view(np.uint32)[0])
kv_field = reader.fields["granitehybrid.attention.head_count_kv"]
head_count_kv = [int(kv_field.parts[kv_field.data[i]].view(np.int32)[0]) for i in range(len(kv_field.data))]

expected_kv = [8 if i in {5, 15, 25, 35} else 0 for i in range(n_blocks)]
kv_ok = head_count_kv == expected_kv
tensor_count_ok = len(reader.tensors) == 666

# 2. Raw header + tensor-info digest & drift check
hasher = hashlib.sha256()
with open(local_gguf, "rb") as f:
    hasher.update(f.read(65536))  # header & KV metadata block
for t in reader.tensors[:16]:     # sample first 16 tensor headers/shapes
    hasher.update(t.name.encode("utf-8"))
    hasher.update(t.shape.tobytes())
    hasher.update(np.array([int(t.tensor_type)], dtype=np.int32).tobytes())
digest = hasher.hexdigest()

drift_status = "BASELINE_CREATED"
max_drift = 0.0
if REF_RECORD.exists():
    ref = json.loads(REF_RECORD.read_text())
    max_drift = 0.0 if digest == ref["digest"] else 1.0
    drift_status = "PASS_ZERO_DRIFT" if max_drift == 0.0 else "DRIFT_DETECTED"
else:
    REF_RECORD.parent.mkdir(parents=True, exist_ok=True)
    REF_RECORD.write_text(json.dumps({"repo_id": REPO_ID, "digest": digest, "tensor_count": len(reader.tensors)}, indent=2))

print(f"File: {local_gguf} ({os.path.getsize(local_gguf) / 1e9:.2f} GB)")
print(f"Arch: {arch} | Blocks: {n_blocks} | Tensors: {len(reader.tensors)} (ok={tensor_count_ok})")
print(f"head_count_kv match ({expected_kv[5]}/{expected_kv[15]}/{expected_kv[25]}/{expected_kv[35]}): {kv_ok}")
print(f"SHA-256 prefix: {digest[:16]}... | Drift: {drift_status} (max_drift={max_drift})")

File: /root/.cache/huggingface/hub/models--frankmorales2020--granite-4.0-h-small-topo-triton-Q8_0-GGUF/snapshots/af9b847e123daa4fa75acbd43a75983c5201d094/granite-4.0-h-small-topo-triton-Q8_0.gguf (34.26 GB)
Arch: granitehybrid | Blocks: 40 | Tensors: 666 (ok=True)
head_count_kv match (8/8/8/8): True
SHA-256 prefix: 1c585796cda8f95c... | Drift: BASELINE_CREATED (max_drift=0.0)


In [3]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/

total 452G
drwx------  2 root root 4.0K Nov 23 15:09 .
-rw-------  1 root root  180 Nov 23 15:09 granite_4_0_h_small_q8_0_audit.json
-rw-------  1 root root  32G Nov 23 14:18 granite-4.0-h-small-topo-triton-Q8_0-patched.gguf
drwx------ 51 root root 4.0K Nov 23 12:42 ..
-rw-------  1 root root  32G Nov 23 11:42 granite-4.0-h-small-topo-triton-Q8_0.gguf
-rw-------  1 root root  16G Nov 23 05:21 kimi-vl-ft-Q8_0.gguf
-rw-------  1 root root 3.7K Nov 22 15:34 GLM-4-32B-0414-topo-triton-Q8_0_audit.json
-rw-------  1 root root  33G Nov 22 14:19 GLM-4-32B-0414-topo-triton-Q8_0.gguf
-rw-------  1 root root 8.9K Nov 21 14:30 somala_drive_gguf_audit.json
-rw-------  1 root root  26G Nov 21 05:08 Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf
-rw-------  1 root root 7.2G Nov 19 03:18 mistral-7b-text-to-sql-topo-Q8_0.gguf
-rw-------  1 root root  27G Nov 14 20:14 qwen3.8-27b-topocbp-Q8_0.gguf
-rw-------  1 root root  11G Nov 12 06:55 deepseek_moe_16b_topocbp_q4_k_m.gguf
-rw-------  1 root root  70G 